# 3. Creare e usare un analyzer custom

Definisci i campi da estrarre da un conto hotel e dalla relativa conferma di pagamento, crea un analyzer custom e analizza il PDF di due pagine con il suo ID. A differenza dell'esercizio 2, lo schema non e quello dell'analyzer prebuilt hotel.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, documento e definizione JSON dell'analyzer.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [2]:
load_dotenv(r"..\.env", override=True)


True

In [3]:
DOCUMENTO = Path(r"..\file\esercizio3.pdf")
document_bytes = DOCUMENTO.read_bytes()


### Schema custom

La definizione [ricevuta_hotel.json](../modelli/ricevuta_hotel.json) usa `prebuilt-document` come base e specifica campi e descrizioni scelti da noi: hotel, ospite, numero ricevuta, totale soggiorno, valuta e importo pagato. Puoi modificarli prima di creare l'analyzer.

In [4]:
definizione = json.loads(
    Path(r"..\modelli\ricevuta_hotel.json").read_text(encoding="utf-8")
)
analyzer_id = "hotel_custom"
print(json.dumps(definizione["fieldSchema"], ensure_ascii=False, indent=2))

{
  "name": "ricevuta_hotel",
  "fields": {
    "Hotel": {
      "type": "string",
      "method": "extract",
      "description": "Nome dell'hotel che emette la ricevuta."
    },
    "Ospite": {
      "type": "string",
      "method": "extract",
      "description": "Nome completo dell'ospite."
    },
    "NumeroRicevuta": {
      "type": "string",
      "method": "extract",
      "description": "Numero della ricevuta del soggiorno (folio)."
    },
    "TotaleSoggiorno": {
      "type": "number",
      "method": "extract",
      "description": "Totale degli addebiti del soggiorno. Non sommare nuovamente l'importo riportato nella conferma di pagamento."
    },
    "Valuta": {
      "type": "string",
      "method": "extract",
      "description": "Codice della valuta degli importi del soggiorno."
    },
    "ImportoPagato": {
      "type": "number",
      "method": "extract",
      "description": "Importo pagato per il soggiorno, riportato nella conferma di pagamento."
    }
  }
}


## 2. Creazione del client

Crea credenziale e client, da usare nelle celle successive.

In [5]:
credential = DefaultAzureCredential()
# Crea il client usando endpoint CU e credenziale Azure corrente.
client = ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"],
    credential=credential,
)


## 3. Creazione dell'analyzer custom

Registra la definizione nel servizio con ID `hotel_custom`. `allow_replace=True` aggiorna l'analyzer del laboratorio se esiste gia; `.result()` attende che la creazione termini.

In [6]:
creazione_completata = False
try:
    analyzer = client.begin_create_analyzer(
        analyzer_id=analyzer_id,
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
    creazione_completata = True
finally:
    if not creazione_completata:
        try:
            client.close()
        finally:
            credential.close()

## 4. Analisi con l'analyzer custom

Passa l'ID stringa `hotel_custom`, non l'oggetto `analyzer`, all'operazione di analisi. Al termine l'analyzer temporaneo viene eliminato e le risorse locali vengono chiuse, anche in caso di errore. Per ripetere l'analisi, riesegui anche le celle di creazione del client e dell'analyzer.

In [7]:
# Usa lo stesso ID scelto durante la creazione dell'analyzer.
try:
    result = client.begin_analyze_binary(
        analyzer_id=analyzer_id,
        binary_input=document_bytes,
    ).result()
finally:
    try:
        client.delete_analyzer(analyzer_id=analyzer_id)
    finally:
        try:
            client.close()
        finally:
            credential.close()


## 5. Risultato

Mostra il Markdown e i campi dello schema custom estratti per ogni contenuto rilevato. Confronta i nomi dei campi con lo schema prebuilt dell'esercizio 2.

In [8]:
for content in result.contents:
    print(content.markdown)
    fields = content.as_dict()["fields"]
    print(json.dumps(fields, ensure_ascii=False, indent=2))

# HOTEL ESEMPIO

Guest folio / Ricevuta soggiorno | H-3001

123 Example Avenue, Seattle, WA 98101, USA

Guest / Ospite: Mario Rossi
Room / Camera: 204
Arrival: 2026-09-01 Departure: 2026-09-04

Currency: USD | Folio status: CLOSED


## Charges and payments


<table>
<tr>
<th>Date</th>
<th>Description</th>
<th>Charges USD</th>
<th>Credits USD</th>
</tr>
<tr>
<td>2026-09-01</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-02</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-03</td>
<td>Room accommodation</td>
<td>500.00</td>
<td></td>
</tr>
<tr>
<td>2026-09-04</td>
<td>Occupancy tax</td>
<td>87.10</td>
<td></td>
</tr>
<tr>
<td>2026-09-04</td>
<td>Card payment</td>
<td></td>
<td>1587.10</td>
</tr>
</table>


Total charges: USD 1587.10

Total payments: USD 1587.10

Balance due: USD 0.00 - PAID IN FULL

Continued on the next page: settlement details for the same folio.

<!-- PageFooter: DATI SINTETICI / TRAINING SAMPLE - Non v